# Surveying Computations: Distances Between Consecutive Points

This Google Colab notebook reads a point-coordinate file from Google Drive and calculates the **horizontal distance** between every consecutive pair of points.

Expected input columns are `P,N,E,Z`: point name, Northing, Easting, and elevation. The horizontal distance uses only Northing and Easting; elevation is retained for reference.

Formula: `D = sqrt((N2 - N1)^2 + (E2 - E1)^2)`.

Coordinates must use a projected coordinate system and the same linear unit (for example, metres).

## Step 1 — Connect Google Drive
Run this cell and authorize access to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Load your point file

Your uploaded sample is a comma-separated text file named `Survey_Points.txt`, with columns `P,N,E,Z`. Copy it into a folder in Google Drive, then set the path below. If your file is a CSV instead, the same code works as long as it uses comma-separated columns.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# EDIT this path to match the file location in your Google Drive
file_path = '/content/drive/MyDrive/Survey_Points.txt'

if not Path(file_path).exists():
    raise FileNotFoundError(
        f'File not found: {file_path}\n'
        'Edit file_path to the actual location of Survey_Points.txt or your CSV file.'
    )

# The provided sample is comma-separated, even though its extension is .txt
points = pd.read_csv(file_path, sep=',', encoding='utf-8-sig')
points.columns = points.columns.astype(str).str.strip()

print(f'Loaded {len(points)} points.')
print('Columns:', list(points.columns))
display(points.head())

## Step 3 — Validate coordinate columns

The sample file uses `P` for point ID, `N` for Northing, `E` for Easting, and `Z` for elevation. Change the column names below only if your file uses different headers.

In [ ]:
point_col = 'P'
northing_col = 'N'
easting_col = 'E'
elevation_col = 'Z'  # retained in the input table; not used for horizontal distance

required = [point_col, northing_col, easting_col]
missing = [c for c in required if c not in points.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}. Available columns: {list(points.columns)}')

work = points.copy()
work[northing_col] = pd.to_numeric(work[northing_col], errors='coerce')
work[easting_col] = pd.to_numeric(work[easting_col], errors='coerce')

invalid = work[work[[northing_col, easting_col]].isna().any(axis=1)]
if not invalid.empty:
    print(f'Warning: excluding {len(invalid)} row(s) with missing or non-numeric coordinates.')
    display(invalid)

work = work.dropna(subset=[northing_col, easting_col]).reset_index(drop=True)
if len(work) < 2:
    raise ValueError('At least two valid points are required.')

display(work)

## Step 4 — Calculate distances between consecutive points

Points are processed in the same order as the rows in the input file. For each pair, the horizontal distance is calculated from Northing and Easting.

In [ ]:
N1 = work[northing_col].iloc[:-1].to_numpy(dtype=float)
N2 = work[northing_col].iloc[1:].to_numpy(dtype=float)
E1 = work[easting_col].iloc[:-1].to_numpy(dtype=float)
E2 = work[easting_col].iloc[1:].to_numpy(dtype=float)

results = pd.DataFrame({
    'From_Point': work[point_col].iloc[:-1].astype(str).to_numpy(),
    'To_Point': work[point_col].iloc[1:].astype(str).to_numpy(),
    'N1': N1,
    'E1': E1,
    'N2': N2,
    'E2': E2,
    'Delta_N': N2 - N1,
    'Delta_E': E2 - E1,
    'Horizontal_Distance': np.hypot(N2 - N1, E2 - E1)
})

display(results.round(3))
print(f'Total consecutive segments: {len(results)}')
print(f'Total horizontal length: {results["Horizontal_Distance"].sum():.3f} coordinate units')

## Step 5 — Export the results

The output is saved as a CSV and downloaded to your computer.

In [ ]:
output_path = '/content/Consecutive_Point_Distances.csv'
results.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Results saved to {output_path}')

from google.colab import files
files.download(output_path)